## Convert Wflow staticmaps netcdf to raster files

In order to inspect or (manually) modify Wflow staticmaps it is convenient to export the maps to a raster format. Here we show how to read the model maps and save to a so-called mapstack (i.e.: a set of raster files with identical grid) using HydroMT.  

### Load dependencies

In [1]:
import shutil
from os.path import join, exists
from os import listdir
import hydromt
from hydromt_wflow import WflowSbmModel

### Read wflow staticmaps

HydroMT provides an easy method to read the model schematization through the Model API.

In [2]:
root = "wflow_piave_subbasin"
wflow = WflowSbmModel(root, mode="r")
ds = wflow.staticmaps.data  # here the staticmaps netcdf is loaded
print(ds)

2026-03-09 10:41:01,173 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-03-09 10:41:01,174 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from c:\Users\matti\miniconda3\envs\hydromt-wflow\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-03-09 10:41:01,230 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-03-09 10:41:01,232 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from C:/Data/TUD/MSc_CE/Courses/2nd_year/7.CIE5060_Thesis/Codes/MSc_thesis_hydromt_wflow/examples/wflow_piave_subbasin/wflow_sbm.toml.
<xarray.Dataset> Size: 975kB
Dimensions:                               (latitude: 53, longitude: 58,
                                           time: 12, layer: 4)
Coordinates:
  * latitude                              (latitude) float64 424B 46.68 ... 4...
  * longitude                             (longitude

### Write netcdf to mapstack

The raster module provides many raster GIS methods through the **raster** Dataset accessor. To write a Dataset into several raster files (eg one geotiff file per map), one line with code is sufficient using [raster.to_mapstack](https://deltares.github.io/hydromt/latest/_generated/hydromt.gis.Dataset.raster.to_mapstack.html#hydromt.gis.Dataset.raster.to_mapstack). We only need to provide the output folder in which all raster files are saved. The default output format is *GeoTIFF*, but this can be changed with the `driver` argument.

In [3]:
# Name of the folder where the tif files will be saved
updated_staticmaps = "updated_staticmaps"

# First remove the folder if it already exists
if exists(updated_staticmaps):
    shutil.rmtree(updated_staticmaps)

# Save the tif files
ds.raster.to_mapstack(updated_staticmaps)
# Print the content of the folder
listdir(updated_staticmaps)

['gauges_grdc.tif',
 'glacier_fraction.tif',
 'glacier_initial_leq_depth.tif',
 'land_elevation.tif',
 'land_manning_n.tif',
 'land_slope.tif',
 'land_water_fraction.tif',
 'local_drain_direction.tif',
 'meta_glacier_area_id.tif',
 'meta_landuse.tif',
 'meta_reservoir_mean_outflow.tif',
 'meta_soilgrids_ksat_vertical_0.0cm.tif',
 'meta_soilgrids_ksat_vertical_100.0cm.tif',
 'meta_soilgrids_ksat_vertical_15.0cm.tif',
 'meta_soilgrids_ksat_vertical_200.0cm.tif',
 'meta_soilgrids_ksat_vertical_30.0cm.tif',
 'meta_soilgrids_ksat_vertical_5.0cm.tif',
 'meta_soilgrids_ksat_vertical_60.0cm.tif',
 'meta_soil_texture.tif',
 'meta_streamorder.tif',
 'meta_subgrid_area.tif',
 'meta_subgrid_elevation.tif',
 'meta_subgrid_outlet_idx.tif',
 'meta_subgrid_outlet_x.tif',
 'meta_subgrid_outlet_y.tif',
 'meta_upstream_area.tif',
 'outlets.tif',
 'reservoir_area.tif',
 'reservoir_area_id.tif',
 'reservoir_b.tif',
 'reservoir_demand.tif',
 'reservoir_e.tif',
 'reservoir_initial_depth.tif',
 'reservoir_low

Now the model files can easily be inspected and modified for example using QGIS (e.g. with the Serval plugin).

> **Note:** in QGIS, you can also visualize netcdf files but direct modification is not (yet) possible.

### Update a specific map using `setup_grid_from_raster`

Let's say you have updated (manually or not) the `soil_ksat_vertical.tif` static map and that you would like to include it in your model. For demonstration purpose, we will make a copy and name it `soil_ksat_vertical_updated.tif`.

In [4]:
old_map = join(updated_staticmaps, "soil_ksat_vertical.tif")
updated_map = join(updated_staticmaps, "soil_ksat_vertical_updated.tif")
shutil.copyfile(old_map, updated_map)

'updated_staticmaps\\soil_ksat_vertical_updated.tif'

You will need to find the `wflow variable` that corresponds to the map you updated, which in the case of `soil_ksat_vertical.tif` is `soil_surface_water__vertical_saturated_hydraulic_conductivity`. For information on the available wflow variables, see the [Wflow documentation](https://deltares.github.io/Wflow.jl/stable/model_docs/intro/)

And then you can simply use [setup_grid_from_raster](https://deltares.github.io/hydromt_wflow/latest/_generated/hydromt_wflow.WflowBaseModel.setup_grid_from_raster.html).

In the below example, we will use python to update our model. If you wish to update via command line, the steps are:

1. Create a data catalog entry for the maps in the "updated_staticmaps" folder
2. Prepare a hydromt config file with the steps: `setup_grid_from_raster`, `statimaps.write` and `config.write`
3. Call the `hydromt update` command line re-using the catalog and configuration file you created.

In [5]:
# Instantiate a new WflowSbmModel object in read/write mode (r+) to be able to update it
wflow = WflowSbmModel(root, mode="r+")

# `KsatVer` corresponds to the `soil_surface_water__vertical_saturated_hydraulic_conductivity` variable
wflow_variable = "soil_surface_water__vertical_saturated_hydraulic_conductivity"
config_opt = wflow.config.get_value(f"input.static.{wflow_variable}")
print(f"Before setup_grid_from_raster: {config_opt}")

# Path to the (updated) `soil_ksat_vertical_updated.tif`
updated_map = join(updated_staticmaps, "soil_ksat_vertical_updated.tif")

# Update the model KsatVer map with setup_grid_from_raster
wflow.setup_grid_from_raster(
    raster_fn=updated_map, 
    reproject_method='nearest',
    variables=["soil_ksat_vertical_updated"], 
    wflow_variables=[wflow_variable],
    
)
config_opt = wflow.config.get_value(f"input.static.{wflow_variable}")
print(f"After setup_grid_from_raster: {config_opt}")

# Write the updated grid and config files to new files
wflow.staticmaps.write(filename="staticmaps_updated.nc")
wflow.config.write(filename="wflow_sbm_updated.toml")


2026-03-09 10:47:34,906 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-03-09 10:47:34,909 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from c:\Users\matti\miniconda3\envs\hydromt-wflow\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-03-09 10:47:34,944 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-03-09 10:47:34,946 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from C:/Data/TUD/MSc_CE/Courses/2nd_year/7.CIE5060_Thesis/Codes/MSc_thesis_hydromt_wflow/examples/wflow_piave_subbasin/wflow_sbm.toml.
Before setup_grid_from_raster: soil_ksat_vertical
2026-03-09 10:47:34,949 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Preparing grid data from raster source updated_staticmaps\soil_ksat_vertical_updated.tif
2026-03-09 10:47:36,159 - hydromt.data_catalog.sources.data_source - data_source - INFO - 